# Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import MinMaxScaler

# Create the sample data (with missing values and an outlier)

In [2]:
data = {
    'Age':        [25, 30, np.nan, 45, 28, 35, 50, np.nan, 22, 40, 29, 33],
    'Salary':     [30000, 45000, 38000, np.nan, 32000, 52000, 60000, 41000, 28000, 500000, 36000, 47000],
    'Experience': [2, 6, 4, 20, 3, 10, 25, np.nan, 1, 15, 4, 8],
    'Department': ['HR', 'IT', 'IT', 'Finance', np.nan, 'IT', 'Finance', 'HR', 'HR', 'IT', 'Finance', 'IT']
}
df = pd.DataFrame(data)
df

,Age,Salary,Experience,Department
0,25.0,30000.0,2.0,HR
1,30.0,45000.0,6.0,IT
2,NaN,38000.0,4.0,IT
3,45.0,NaN,20.0,Finance
4,28.0,32000.0,3.0,NaN
5,35.0,52000.0,10.0,IT
6,50.0,60000.0,25.0,Finance
7,NaN,41000.0,NaN,HR
8,22.0,28000.0,1.0,HR
9,40.0,500000.0,15.0,IT


# **Step 1a**: Detect missing values

In [3]:
print(df.isnull().sum())
print("\nPercentage missing:\n", (df.isnull().mean() * 100).round(2))

Age           2
Salary        1
Experience    1
Department    1
dtype: int64

Percentage missing:
 Age           16.67
Salary         8.33
Experience     8.33
Department     8.33
dtype: float64


# S**tep 1b**: Fill missing values

In [5]:
df_clean = df.copy()

df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].mean())                 # mean
df_clean['Salary'] = df_clean['Salary'].fillna(df_clean['Salary'].median())      # median
df_clean['Experience'] = df_clean['Experience'].fillna(df_clean['Experience'].median())
df_clean['Department'] = df_clean['Department'].fillna(df_clean['Department'].mode()[0])  # mode

print(df_clean.isnull().sum())
df_clean

Age           0
Salary        0
Experience    0
Department    0
dtype: int64


,Age,Salary,Experience,Department
0,25.0,30000.0,2.0,HR
1,30.0,45000.0,6.0,IT
2,33.7,38000.0,4.0,IT
3,45.0,41000.0,20.0,Finance
4,28.0,32000.0,3.0,IT
5,35.0,52000.0,10.0,IT
6,50.0,60000.0,25.0,Finance
7,33.7,41000.0,6.0,HR
8,22.0,28000.0,1.0,HR
9,40.0,500000.0,15.0,IT


# **Step 2a**: Outliers using the IQR method

In [11]:
Q1 = df_clean['Salary'].quantile(0.25)
Q3 = df_clean['Salary'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Q1={Q1}, Q3={Q3}, IQR={IQR}")
print(f"Lower bound={lower}, Upper bound={upper}")

outliers_iqr = df_clean[(df_clean['Salary'] < lower) | (df_clean['Salary'] > upper)]
outliers_iqr

Q1=35000.0, Q3=48250.0, IQR=13250.0
Lower bound=15125.0, Upper bound=68125.0


,Age,Salary,Experience,Department,Salary_zscore
9,40.0,500000.0,15.0,IT,3.308595


# **Step 2b**: Treat outliers by capping

In [12]:
df_clean['Salary_capped'] = df_clean['Salary'].clip(lower=lower, upper=upper)
df_clean[['Salary', 'Salary_capped']]

,Salary,Salary_capped
0,30000.0,30000.0
1,45000.0,45000.0
2,38000.0,38000.0
3,41000.0,41000.0
4,32000.0,32000.0
5,52000.0,52000.0
6,60000.0,60000.0
7,41000.0,41000.0
8,28000.0,28000.0
9,500000.0,68125.0


# Convert numerical data to categorical

In [13]:
# Fixed ranges (domain-defined)
df_clean['Age_Group'] = pd.cut(df_clean['Age'],
                               bins=[0, 25, 35, 50, 100],
                               labels=['Young', 'Early Career', 'Mid Career', 'Senior'])

# Equal-frequency groups (quantile-based)
df_clean['Salary_Level'] = pd.qcut(df_clean['Salary_capped'], q=3,
                                   labels=['Low', 'Medium', 'High'])

df_clean[['Age', 'Age_Group', 'Salary_capped', 'Salary_Level']]

,Age,Age_Group,Salary_capped,Salary_Level
0,25.0,Young,30000.0,Low
1,30.0,Early Career,45000.0,Medium
2,33.7,Early Career,38000.0,Medium
3,45.0,Mid Career,41000.0,Medium
4,28.0,Early Career,32000.0,Low
5,35.0,Early Career,52000.0,High
6,50.0,Mid Career,60000.0,High
7,33.7,Early Career,41000.0,Medium
8,22.0,Young,28000.0,Low
9,40.0,Mid Career,68125.0,High


# **Step 4a**: Create bins (equal-depth)

In [14]:
df_clean['Salary_Bin'] = pd.qcut(df_clean['Salary_capped'], q=4, labels=False)
print(df_clean.groupby('Salary_Bin')['Salary_capped'].apply(list))

Salary_Bin
0             [30000.0, 32000.0, 28000.0]
1    [38000.0, 41000.0, 41000.0, 36000.0]
2                      [45000.0, 47000.0]
3             [52000.0, 60000.0, 68125.0]
Name: Salary_capped, dtype: object


# **Step 4b**: Smooth data using the bins

In [15]:
g = df_clean.groupby('Salary_Bin')['Salary_capped']

# Smoothing by bin means and medians
df_clean['Smooth_Mean'] = g.transform('mean')
df_clean['Smooth_Median'] = g.transform('median')

# Smoothing by bin boundaries (replace each value with the closer of min/max)
bin_min = g.transform('min')
bin_max = g.transform('max')
df_clean['Smooth_Boundary'] = np.where(
    (df_clean['Salary_capped'] - bin_min) <= (bin_max - df_clean['Salary_capped']),
    bin_min, bin_max)

df_clean[['Salary_capped', 'Salary_Bin', 'Smooth_Mean', 'Smooth_Median', 'Smooth_Boundary']] \
    .sort_values('Salary_capped')

,Salary_capped,Salary_Bin,Smooth_Mean,Smooth_Median,Smooth_Boundary
8,28000.0,0,30000.000000,30000.0,28000.0
0,30000.0,0,30000.000000,30000.0,28000.0
4,32000.0,0,30000.000000,30000.0,32000.0
10,36000.0,1,39000.000000,39500.0,36000.0
2,38000.0,1,39000.000000,39500.0,36000.0
3,41000.0,1,39000.000000,39500.0,41000.0
7,41000.0,1,39000.000000,39500.0,41000.0
1,45000.0,2,46000.000000,46000.0,45000.0
11,47000.0,2,46000.000000,46000.0,47000.0
5,52000.0,3,60041.666667,60000.0,52000.0


# **Step 5a**: Min-max normalization (manual formula)

In [16]:
cols = ['Age', 'Salary_capped', 'Experience']

df_norm = df_clean.copy()
for c in cols:
    df_norm[c + '_norm'] = (df_clean[c] - df_clean[c].min()) / (df_clean[c].max() - df_clean[c].min())

df_norm[[c + '_norm' for c in cols]]

,Age_norm,Salary_capped_norm,Experience_norm
0,0.107143,0.049844,0.041667
1,0.285714,0.423676,0.208333
2,0.417857,0.249221,0.125000
3,0.821429,0.323988,0.791667
4,0.214286,0.099688,0.083333
5,0.464286,0.598131,0.375000
6,1.000000,0.797508,1.000000
7,0.417857,0.323988,0.208333
8,0.000000,0.000000,0.000000
9,0.642857,1.000000,0.583333


# **Step 5b**: Same result using scikit-learn

In [17]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(df_clean[cols])
df_sklearn = pd.DataFrame(scaled, columns=[c + '_scaled' for c in cols])

print(df_sklearn.round(3))
print("\nMin:", df_sklearn.min().values, " Max:", df_sklearn.max().values)

    Age_scaled  Salary_capped_scaled  Experience_scaled
0        0.107                 0.050              0.042
1        0.286                 0.424              0.208
2        0.418                 0.249              0.125
3        0.821                 0.324              0.792
4        0.214                 0.100              0.083
5        0.464                 0.598              0.375
6        1.000                 0.798              1.000
7        0.418                 0.324              0.208
8        0.000                 0.000              0.000
9        0.643                 1.000              0.583
10       0.250                 0.199              0.125
11       0.393                 0.474              0.292

Min: [0. 0. 0.]  Max: [1. 1. 1.]


# Final preprocessed dataset

In [18]:
final_df = df_norm.drop(columns=['Salary_zscore'])
final_df

,Age,Salary,Experience,Department,Salary_capped,Age_Group,Salary_Level,Salary_Bin,Smooth_Mean,Smooth_Median,Smooth_Boundary,Age_norm,Salary_capped_norm,Experience_norm
0,25.0,30000.0,2.0,HR,30000.0,Young,Low,0,30000.000000,30000.0,28000.0,0.107143,0.049844,0.041667
1,30.0,45000.0,6.0,IT,45000.0,Early Career,Medium,2,46000.000000,46000.0,45000.0,0.285714,0.423676,0.208333
2,33.7,38000.0,4.0,IT,38000.0,Early Career,Medium,1,39000.000000,39500.0,36000.0,0.417857,0.249221,0.125000
3,45.0,41000.0,20.0,Finance,41000.0,Mid Career,Medium,1,39000.000000,39500.0,41000.0,0.821429,0.323988,0.791667
4,28.0,32000.0,3.0,IT,32000.0,Early Career,Low,0,30000.000000,30000.0,32000.0,0.214286,0.099688,0.083333
5,35.0,52000.0,10.0,IT,52000.0,Early Career,High,3,60041.666667,60000.0,52000.0,0.464286,0.598131,0.375000
6,50.0,60000.0,25.0,Finance,60000.0,Mid Career,High,3,60041.666667,60000.0,52000.0,1.000000,0.797508,1.000000
7,33.7,41000.0,6.0,HR,41000.0,Early Career,Medium,1,39000.000000,39500.0,41000.0,0.417857,0.323988,0.208333
8,22.0,28000.0,1.0,HR,28000.0,Young,Low,0,30000.000000,30000.0,28000.0,0.000000,0.000000,0.000000
9,40.0,500000.0,15.0,IT,68125.0,Mid Career,High,3,60041.666667,60000.0,68125.0,0.642857,1.000000,0.583333
